# Odoo Development with Jupyter

Este notebook te ayuda a conectarte con tu instancia de Odoo y realizar desarrollo y análisis de datos.

## Configuración inicial

Primero, importamos las librerías necesarias y configuramos la conexión con Odoo.

In [ ]:
import os
import xmlrpc.client
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import psycopg2
from sqlalchemy import create_engine

# Configuración de visualización
plt.style.use('default')
sns.set_palette("husl")
%matplotlib inline

print("✅ Librerías importadas correctamente")

## Conexión con Odoo via XML-RPC

Configuramos la conexión con nuestra instancia de Odoo.

In [ ]:
# Configuración de Odoo (ajustar según tu proyecto)
ODOO_URL = 'http://odoo:8069'  # Usar nombre del servicio Docker
ODOO_DB = os.environ.get('DB_NAME', 'odoo')  # Nombre de la base de datos
ODOO_USERNAME = 'admin'  # Usuario admin por defecto
ODOO_PASSWORD = 'admin'  # Contraseña admin por defecto

print(f"Conectando a Odoo en: {ODOO_URL}")
print(f"Base de datos: {ODOO_DB}")

# Crear conexiones XML-RPC
try:
    common = xmlrpc.client.ServerProxy(f'{ODOO_URL}/xmlrpc/2/common')
    object_proxy = xmlrpc.client.ServerProxy(f'{ODOO_URL}/xmlrpc/2/object')
    
    # Autenticar
    uid = common.authenticate(ODOO_DB, ODOO_USERNAME, ODOO_PASSWORD, {})
    
    if uid:
        print(f"✅ Conectado exitosamente a Odoo como usuario ID: {uid}")
        
        # Función helper para ejecutar llamadas a Odoo
        def odoo_execute(model, method, *args, **kwargs):
            return object_proxy.execute_kw(ODOO_DB, uid, ODOO_PASSWORD, model, method, args, kwargs)
        
        # Función para buscar registros
        def odoo_search_read(model, domain=[], fields=[], limit=None, offset=0, order=None):
            kwargs = {'fields': fields, 'offset': offset}
            if limit:
                kwargs['limit'] = limit
            if order:
                kwargs['order'] = order
            return odoo_execute(model, 'search_read', domain, kwargs)
        
        print("📚 Funciones de helper creadas: odoo_execute(), odoo_search_read()")
    else:
        print("❌ Error de autenticación")
        
except Exception as e:
    print(f"❌ Error conectando a Odoo: {e}")
    print("Asegúrate de que Odoo esté ejecutándose en el contenedor 'odoo'")

## Conexión directa a PostgreSQL

Para análisis más complejos, puedes conectarte directamente a la base de datos.

In [ ]:
# Configuración de PostgreSQL
DB_HOST = os.environ.get('DB_HOST', 'postgres')
DB_PORT = os.environ.get('DB_PORT', '5432')
DB_USER = os.environ.get('DB_USER', 'odoo')
DB_PASSWORD = os.environ.get('DB_PASSWORD', 'odoo')
DB_NAME = os.environ.get('DB_NAME', 'odoo')

# Crear engine de SQLAlchemy
try:
    engine = create_engine(f'postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}')
    
    # Probar conexión
    with engine.connect() as conn:
        result = conn.execute("SELECT version();")
        version = result.fetchone()[0]
        print(f"✅ Conectado a PostgreSQL: {version.split(',')[0]}")
        
    # Función helper para queries SQL
    def query_to_dataframe(query):
        return pd.read_sql(query, engine)
        
    print("📊 Función helper creada: query_to_dataframe()")
    
except Exception as e:
    print(f"❌ Error conectando a PostgreSQL: {e}")

## Ejemplos de uso

### 1. Listar usuarios del sistema

In [ ]:
if 'odoo_search_read' in globals():
    try:
        # Obtener usuarios activos
        users = odoo_search_read(
            'res.users',
            domain=[('active', '=', True)],
            fields=['name', 'login', 'email', 'create_date'],
            limit=10
        )
        
        if users:
            df_users = pd.DataFrame(users)
            print(f"👥 Encontrados {len(df_users)} usuarios:")
            display(df_users)
        else:
            print("No se encontraron usuarios")
    except Exception as e:
        print(f"Error obteniendo usuarios: {e}")
else:
    print("⚠️  Primero ejecuta la celda de conexión a Odoo")

### 2. Análisis de tablas de la base de datos

In [ ]:
if 'query_to_dataframe' in globals():
    try:
        # Obtener información de las tablas principales
        tables_query = """
        SELECT 
            schemaname,
            tablename,
            n_tup_ins as inserts,
            n_tup_upd as updates,
            n_tup_del as deletes,
            n_live_tup as live_rows
        FROM pg_stat_user_tables 
        WHERE schemaname = 'public'
        AND tablename LIKE 'res_%' OR tablename LIKE 'ir_%'
        ORDER BY n_live_tup DESC 
        LIMIT 15;
        """
        
        df_tables = query_to_dataframe(tables_query)
        
        if not df_tables.empty:
            print("📋 Estadísticas de las tablas principales:")
            display(df_tables)
            
            # Gráfico de barras de registros por tabla
            plt.figure(figsize=(12, 6))
            plt.bar(df_tables['tablename'], df_tables['live_rows'])
            plt.title('Número de registros por tabla')
            plt.xlabel('Tabla')
            plt.ylabel('Número de registros')
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()
        else:
            print("No se encontraron datos de tablas")
    except Exception as e:
        print(f"Error analizando tablas: {e}")
else:
    print("⚠️  Primero ejecuta la celda de conexión a PostgreSQL")

### 3. Espacio para tu análisis personalizado

Usa este espacio para escribir tus propias consultas y análisis:

In [ ]:
# Tu código aquí
# Ejemplo:
# custom_query = "SELECT * FROM res_partner LIMIT 5;"
# df_partners = query_to_dataframe(custom_query)
# display(df_partners)

print("💡 Escribe tu código de análisis aquí")

## Tips útiles

### Modelos principales de Odoo:
- `res.users` - Usuarios
- `res.partner` - Contactos/Clientes/Proveedores
- `res.company` - Empresas
- `ir.model` - Modelos de datos
- `ir.model.fields` - Campos de modelos
- `mail.message` - Mensajes/Log

### Funciones útiles:
```python
# Buscar registros
records = odoo_search_read('model.name', domain=[('field', '=', 'value')], fields=['field1', 'field2'])

# Crear registro
new_id = odoo_execute('model.name', 'create', {'field1': 'value1', 'field2': 'value2'})

# Actualizar registro
odoo_execute('model.name', 'write', [record_id], {'field1': 'new_value'})

# Query SQL directa
df = query_to_dataframe("SELECT * FROM table_name WHERE condition")
```

### Variables de entorno disponibles:
- `DB_NAME` - Nombre de la base de datos
- `DB_HOST` - Host de PostgreSQL
- `DB_USER` - Usuario de base de datos
- `DB_PASSWORD` - Contraseña de base de datos